🧱 1. Install + imports

In [ ]:
!pip install sentence-transformers faiss-cpu pypdf

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from pypdf import PdfReader
import os

📄 2. Load PDFs with page numbers and headings (& other files code may still need to be added later)

In [ ]:
# 🔴 ADAPT THIS PATH
folder_path = "../data/raw/"

# Simple heading detection heuristic
def extract_heading(text):
    lines = text.split("\n")
    for line in lines:
        if len(line.strip()) > 3 and line.strip().isupper():
            return line.strip()
    return "Unknown"

def load_pdfs_with_pages(folder_path):
    documents = []

    if not os.path.exists(folder_path):
        print(f"❌ Folder not found: {folder_path}")
        return documents

    # Walk recursively to include subfolders
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(".pdf"):
                full_path = os.path.join(root, file)
                try:
                    reader = PdfReader(full_path)
                    for i, page in enumerate(reader.pages):
                        text = page.extract_text()
                        if text and text.strip():
                            heading = extract_heading(text)
                            documents.append({
                                "text": text.strip(),
                                "source": file,
                                "page": i+1,      # Pages are 1-indexed
                                "heading": heading
                            })
                        else:
                            print(f"⚠️ Empty page {i+1} in {file}")
                except Exception as e:
                    print(f"❌ Error reading {file}: {e}")

    return documents

# Load all PDFs
docs = load_pdfs_with_pages(folder_path)
print(f"\n✅ Loaded {len(docs)} pages from PDFs")

✂️ 3. Chunk the text

In [ ]:
# Split text into chunks and keep page + heading
chunk_size = 500  # words

chunks = []
metadata = []

for doc in docs:
    words = doc["text"].split()
    for i in range(0, len(words), chunk_size):
        chunk_text = " ".join(words[i:i+chunk_size])
        chunks.append(chunk_text)
        metadata.append({
            "source": doc["source"],
            "page": doc["page"],
            "heading": doc["heading"]
        })

print(f"✅ Created {len(chunks)} chunks")

🧠 4. Create embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Model (free, local)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

print(f"✅ Embeddings created: {embeddings.shape}")

🔎 5. Store in FAISS

In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"✅ FAISS index ready with {index.ntotal} vectors")

❓ 6. Ask a question (retrieval only)

In [48]:
def search(query, top_k=5, threshold=None):
    # 1️⃣ Encode the query
    query_embedding = model.encode([query]).astype("float32")

    # 2️⃣ Search top_k vectors
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for dist, idx in zip(distances[0], indices[0]):
        if threshold is None or dist < threshold:
            results.append({
                "text": chunks[idx],
                "source": metadata[idx]["source"],
                "page": metadata[idx]["page"],
                "heading": metadata[idx]["heading"],
                "distance": dist
            })
    return results

question = "data science skills"  # <-- Can update test question here
results = search(question, top_k=5)  # threshold optional

# 3️⃣ Display results with preview
for i, r in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source : {r['source']}")
    print(f"Page   : {r['page']}")
    print(f"Heading: {r['heading']}")
    print(f"Distance: {r['distance']:.4f}")
    print(f"Preview: {r['text'][:300]}")  # first 300 chars


--- Result 1 ---
Source : 2019_Evolution-of-CDM-to-CDS-Part-1-Drivers.pdf
Page   : 20
Heading: Unknown
Distance: 0.9759
Preview: The Evolution of Clinical Data Management into Clinical Data Science Society for Clinical Data Management Reflection Paper 20 b) Foundational Clinical Data Management Competencies While the role is evolving, several of the foundational competencies for a Clinical Data Manager remain the same. For ex

--- Result 2 ---
Source : 2020_Evolution-of-CDM-to-CDS-Part-3-Evolution-of-CDM-Role.pdf
Page   : 1
Heading: Unknown
Distance: 1.0530
Preview: The Evolution of Clinical Data Management into Clinical Data Science (Part 3: The evolution of the CDM role) A Reflection Paper on the evolution of CDM skillsets and competencies 31 Aug 2020 Society for Clinical Data Management © Society for Clinical Data Management Our Vision “Leading innovative cl

--- Result 3 ---
Source : SCDM-Position-Paper-Evolution-into-Clinical-to-Data-Science-V9.0.pdf
Page   : 7
Heading: Unknown
D

🤖 7. Add simple answer generation

In [59]:
from transformers import pipeline

generator = pipeline("text-generation", model="NousResearch/Nous-Hermes-13b", device=-1)  # CPU

def answer_from_chunks(question, chunks_metadata):
    context = "\n\n".join([f"[{c['source']} - Page {c['page']}] {c['text']}" for c in chunks_metadata])
    prompt = f"Answer based on the context:\n{context}\nQuestion: {question}\nAnswer:"
    return generator(prompt, max_new_tokens=300)[0]['generated_text']

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Could not extract SentencePiece model from /Users/kristinnuyens/.cache/huggingface/hub/models--NousResearch--Nous-Hermes-13b/snapshots/24e8c03148ffd1f3e469744dfc24ad2ad82848f8/tokenizer.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.

In [ ]:
from transformers import pipeline

# CPU-friendly HF text generation model
generator = pipeline(
    "text-generation",
    model="NousResearch/Nous-Hermes-13b",  # CPU-friendly instruct model
    device=-1  # -1 = CPU
)

def answer_from_chunks(question, chunks_metadata, max_new_tokens=300, top_chunks=5):
    # Take only top N chunks
    top_context = chunks_metadata[:top_chunks]
    
    # Prepare context with source/page info
    context = "\n\n".join([f"[{c['source']} - Page {c['page']}] {c['text']}" for c in top_context])
    
    # Prompt for the model
    prompt = (
        f"Answer the question based on the context below. "
        f"Cite PDF and page number if possible.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    
    # Generate answer
    output = generator(prompt, max_new_tokens=max_new_tokens)[0]['generated_text']
    return output